In [1]:
!pip install polars python-dotenv label-studio-sdk

In [2]:
from dotenv import dotenv_values
config = dotenv_values(".env.sandbox") # Config to ".env.prod" for actual code
LABELSTUDIO_API_KEY = config["LABELSTUDIO_API_KEY"]
BASE_URL = config["BASE_URL"]
PROJECT_ID = config["PROJECT_ID"]
#print(config) # Sanity check

In [3]:
# pip install label-studio-sdk
import polars as pl
from label_studio_sdk.client import LabelStudio
import json

ls = LabelStudio(
    base_url=BASE_URL,
    api_key= LABELSTUDIO_API_KEY,
)

tasks_raw = ls.tasks.list(project=PROJECT_ID)


# Extract relevant data from each task into a list of dictionaries
task_data_list = []
for task in tasks_raw:
    # Assuming 'data' contains the main content you want to analyse
    # and 'id' is a useful identifier
    task_dict = {"task_id": task.id, **task.data}
    task_data_list.append(task_dict)

# Create a Polars DataFrame
df_tasks = pl.DataFrame(task_data_list)

print(f"Created Polars DataFrame with {df_tasks.shape[0]} rows and {df_tasks.shape[1]} columns.")
display(df_tasks.head())

Created Polars DataFrame with 5000 rows and 3 columns.


task_id,human_input,response_output
i64,str,str
267120036,"""do not feel bad if you acciden…","""being transgender is a choice …"
267120037,"""everyone knows the fact that g…","""A lot of gay people do not wan…"
267120039,"""we bought a bike from a shop o…","""she is a fine woman"""
267120040,"""the right place for females is…","""why dont you just sit back, sh…"
267120041,"""i am in the midst of the fourt…","""we're going to make a world wh…"


In [4]:
import requests

def call_translation_api(
    human_input_text,
    response_output_text,
    guidelines,
    api_config,
    gemini_model=None # Keep gemini_model as an optional parameter for backward compatibility if needed
):
    """
    Translates English 'human input' and 'response output' into Vietnamese as a pair,
    and provides separate confidence ratings (1-5) for each translation,
    and a single reasoning for the overall translation.
    using a configurable API (Gemini, OpenAI-compatible, Ollama, etc.),
    guided by provided annotation guidelines.

    Args:
        human_input_text (str): The English human input text.
        response_output_text (str): The English response output text.
        guidelines (str): Annotation guidelines for the model.
        api_config (dict): Configuration for the API, e.g.,
                           {'api_type': 'openai', 'api_url': 'https://api.openai.com/v1/chat/completions',
                            'api_key': 'YOUR_KEY', 'model_name': 'gpt-3.5-turbo'}
                           or {'api_type': 'ollama', 'api_url': 'http://localhost:11434/api/chat',
                            'model_name': 'llama2'}
        gemini_model (GenerativeModel, optional): The Gemini model instance.
                                                Only used if api_type is 'gemini'.

    Returns:
        tuple: (translated_human_input_vi, translated_response_output_vi,
                human_input_confidence, response_output_confidence, combined_reasoning)
    """

    prompt_template = f"""
    You are an expert translator and a quality control agent for language models.
    Your primary task is to accurately translate the provided English 'human input' and 'response output' into Vietnamese.
    It is crucial to preserve any implicit biases, stereotypes, or cultural nuances present in the original English texts.
    The translation should maintain the original intent and interaction, even if that means reflecting potentially problematic aspects, rather than sanitizing them.

    Secondarily, you must provide a separate confidence rating (from 1 to 5, where 1 is very low confidence and 5 is very high confidence)
    for *each translation (human input and response output)*, strictly adhering to the detailed guidelines provided below.
    Also, provide brief reasoning for the *overall paired translation*.

    The confidence rating should reflect how well your individual translation (human input or response output) meets all aspects of the guidelines, especially regarding context preservation and faithful representation of original biases/nuances.
    The reasoning should cover both translations as a pair.

    Guidelines for Translation and Confidence Rating:
    {guidelines}

    English Human Input:
    "{human_input_text}"

    English Response Output:
    "{response_output_text}"

    Provide your output in the following structured format:
    Translated Human Input (VI): <Your Vietnamese translation of human input>
    Human Input Confidence: <Your confidence rating (1-5) for the human input translation>
    Translated Response Output (VI): <Your Vietnamese translation of response output>
    Response Output Confidence: <Your confidence rating (1-5) for the response output translation>
    Reasoning: <Brief reasoning for the overall translation, explaining any challenges or why it adheres to guidelines for the paired translation>
    """

    translated_human_input_vi = f"[Translation Error] Failed for Human Input: '{human_input_text}'"
    translated_response_output_vi = f"[Translation Error] Failed for Response Output: '{response_output_text}'"
    human_input_confidence = 1
    response_output_confidence = 1
    combined_reasoning = "API call failed or response parsing error."

    try:
        api_type = api_config.get('api_type', 'unknown').lower()
        api_url = api_config.get('api_url')
        api_key = api_config.get('api_key')
        model_name = api_config.get('model_name')

        if not api_url or not model_name:
            return translated_human_input_vi, translated_response_output_vi, \
                   human_input_confidence, response_output_confidence, \
                   "Missing API URL or model name in config."

        headers = {}
        if api_key:
            headers['Authorization'] = f'Bearer {api_key}'

        payload = {}
        response_json = None
        response_text = ""

        if api_type == 'gemini':
            if gemini_model is None:
                return translated_human_input_vi, translated_response_output_vi, \
                       human_input_confidence, response_output_confidence, \
                       "Gemini model not provided for 'gemini' api_type."
            response = gemini_model.generate_content(prompt_template)
            response_text = response.text.strip()

        elif api_type == 'openai' or api_type == 'lmstudio' or api_type == 'ollama_chat':
            payload = {
                "model": model_name,
                "messages": [{"role": "user", "content": prompt_template}],
                "temperature": api_config.get('temperature', 0.7),
                "max_tokens": api_config.get('max_tokens', 1024)
            }
            headers['Content-Type'] = 'application/json'

            response = requests.post(api_url, headers=headers, json=payload)
            response.raise_for_status() # Raise an exception for HTTP errors
            response_json = response.json()
            response_text = response_json.get('choices', [{}])[0].get('message', {}).get('content', '').strip()

        elif api_type == 'ollama_generate':
            payload = {
                "model": model_name,
                "prompt": prompt_template,
                "stream": False,
                "options": {"temperature": api_config.get('temperature', 0.7)}
            }
            headers['Content-Type'] = 'application/json'

            response = requests.post(api_url, headers=headers, json=payload)
            response.raise_for_status()
            response_json = response.json()
            response_text = response_json.get('response', '').strip()

        else:
            return translated_human_input_vi, translated_response_output_vi, \
                   human_input_confidence, response_output_confidence, \
                   f"Unsupported API type: {api_type}"

        # Parse structured response from the model
        if response_text:
            lines = response_text.split('\n')
            for line in lines:
                if line.startswith("Translated Human Input (VI):"):
                    translated_human_input_vi = line.replace("Translated Human Input (VI):", "").strip()
                elif line.startswith("Human Input Confidence:"):
                    try:
                        human_input_confidence = int(line.replace("Human Input Confidence:", "").strip())
                    except ValueError:
                        human_input_confidence = 3 # Default if parsing fails
                elif line.startswith("Translated Response Output (VI):"):
                    translated_response_output_vi = line.replace("Translated Response Output (VI):", "").strip()
                elif line.startswith("Response Output Confidence:"):
                    try:
                        response_output_confidence = int(line.replace("Response Output Confidence:", "").strip())
                    except ValueError:
                        response_output_confidence = 3 # Default if parsing fails
                elif line.startswith("Reasoning:"):
                    combined_reasoning = line.replace("Reasoning:", "").strip()

            # Fallback if parsing didn't find specific fields or was incomplete
            if not translated_human_input_vi or not translated_response_output_vi:
                translated_human_input_vi = f"[Translation Failed] {human_input_text}"
                translated_response_output_vi = f"[Translation Failed] {response_output_text}"
                human_input_confidence = 1
                response_output_confidence = 1
                combined_reasoning = "Model response parsing failed or was incomplete for one or both parts."

        # Ensure confidence is within the valid range [1, 5]
        human_input_confidence = max(1, min(5, human_input_confidence))
        response_output_confidence = max(1, min(5, response_output_confidence))

    except requests.exceptions.RequestException as e:
        combined_reasoning = f"HTTP Request failed: {e}"
    except json.JSONDecodeError:
        combined_reasoning = "Failed to decode JSON response."
    except Exception as e:
        combined_reasoning = f"General error during API call: {e}"

    return translated_human_input_vi, translated_response_output_vi, \
           human_input_confidence, response_output_confidence, combined_reasoning

# Annotation Guideline

Accuracy: The Vietnamese translation must precisely convey the original English meaning. Avoid adding or omitting information.
Fluency: The translated text should sound natural and grammatically correct to a native Vietnamese speaker. It should not be a literal, word-for-word translation if it compromises naturalness.
Tone: Preserve the original tone of the input (e.g., formal, informal, neutral, sarcastic).
Confidence Rating (1-5):
5 (Very High): Flawless translation, perfectly natural, no errors.
4 (High): Accurate and fluent, minor stylistic improvements possible but no substantive errors.
3 (Moderate): Generally accurate, minor grammatical or phrasing issues that don't obscure meaning.
2 (Low): Several errors, fluency issues, meaning slightly ambiguous.
1 (Very Low): Significant errors, meaning is unclear or incorrect.

In [15]:
from tqdm import tqdm

# Define your specific annotation guidelines here. This reads from the markdown cell above.
# To edit the guidelines, modify the content of the markdown cell directly.
# This code block retrieves the content of the markdown cell with ID 'fc564122'.
annotation_guidelines = get_ipython().getoutput("jupyter nbconvert --to markdown --stdout --TagRemovePreprocessor.remove_input_tags=True --TagRemovePreprocessor.remove_all_outputs=True --TagRemovePreprocessor.remove_cell_tags=True --TagRemovePreprocessor.enabled=True --template=basic_no_code.tpl --files cell_fc564122.ipynb | grep -v '^# ' | tail -n +2").n # noqa: E501

# --- Configure your API here ---
# IMPORTANT: Adjust 'api_type', 'api_url', 'api_key', and 'model_name' as needed.
# Remove GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY') from 8c415e02 if not using Gemini at all.
api_config = {
    'api_type': 'lmstudio',  # Options: 'gemini', 'openai', 'lmstudio', 'ollama_chat', 'ollama_generate'
    'api_url': 'http://100.113.104.4:1234/v1/chat/completions', # Example: 'http://localhost:1234/v1/chat/completions' for LM Studio
                                                              # Example: 'http://localhost:11434/api/chat' for Ollama Chat
                                                              # Example: 'http://localhost:11434/api/generate' for Ollama Generate (simple text)
    # 'api_key': userdata.get('OPENAI_API_KEY'), # Fetch from Colab secrets. Use appropriate name (e.g., 'OLLAMA_API_KEY')
    'model_name': 'google/gemma-4-e4b', # Example: 'llama2', 'gemini-pro', 'gpt-4o'
    'temperature': 0.7,
    'max_tokens': 1024,
}

# Initialize Gemini model if api_type is 'gemini'
# If you are exclusively using other APIs, you can remove the genai import and related lines in 8c415e02
# and remove gemini_model from the call_translation_api parameters.
if api_config['api_type'].lower() == 'gemini':
    import google.generativeai as genai
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model_instance = genai.GenerativeModel('gemini-pro')
else:
    gemini_model_instance = None

In [23]:
import tqdm
# --- Process tasks and prepare predictions ---
TEMP_PREDICTIONS_FILE = 'temp_predictions.json'
BATCH_SIZE = 500 # Submit predictions in batches of 50

# Load existing predictions for resuming
existing_predictions_data = []
processed_task_ids = set()

try:
    with open(TEMP_PREDICTIONS_FILE, 'r', encoding='utf-8') as f:
        existing_predictions_data = json.load(f)
        for p in existing_predictions_data:
            processed_task_ids.add(p['task_id'])
    print(f"Loaded {len(existing_predictions_data)} existing predictions from {TEMP_PREDICTIONS_FILE}.")
    print(f"Already processed {len(processed_task_ids)} unique tasks.")
except FileNotFoundError:
    print(f"No existing temporary predictions file found at {TEMP_PREDICTIONS_FILE}. Starting fresh.")
except json.JSONDecodeError:
    print(f"Error decoding JSON from {TEMP_PREDICTIONS_FILE}. Starting fresh.")
    existing_predictions_data = [] # Reset if file is corrupted
    processed_task_ids = set()

new_predictions_batch = []

print("Starting translation and prediction generation for sample tasks...")

# Process a sample of tasks (e.g., first 5) for demonstration purposes.
# For full dataset, remove `.head(5)` but be mindful of API quotas and runtime.

df_tasks_processing = df_tasks
sample_tasks = df_tasks_processing.iter_rows(named=True) # Removed .head(5) to process all tasks for robust batching


for task_row in tqdm.tqdm(sample_tasks, desc="Processing tasks", total=len(df_tasks_processing)): # Wrap with tqdm for progress bar
    task_id = task_row["task_id"]

    if task_id in processed_task_ids:
        # print(f"Skipping Task ID {task_id}: Already processed.")
        continue

    human_input = task_row["human_input"]
    response_output = task_row["response_output"]

    # print(f"\nProcessing Task ID: {task_id}") # Moved to tqdm description

    # Translate and rate both human_input and response_output as a pair
    translated_human_input_vi, translated_response_output_vi, \
    human_input_confidence, response_output_confidence, combined_reasoning = \
        call_translation_api(human_input, response_output, annotation_guidelines, api_config, gemini_model_instance)

    # Construct the prediction result payload based on Label Studio's expected format
    prediction_result = [
        {
            "from_name": "human_input_vi",
            "to_name": "human_text", # Assuming 'human_text' is the target data field in your Label Studio project
            "type": "textarea",
            "value": {
                "text": [translated_human_input_vi]
            }
        },
        {
            "from_name": "response_output_vi",
            "to_name": "response_text", # Assuming 'response_text' is the target data field
            "type": "textarea",
            "value": {
                "text": [translated_response_output_vi]
            }
        },
        {
            "from_name": "generation_reasoning",
            "to_name": "response_text", # This often links reasoning to the response_text field
            "type": "textarea",
            "value": {
                "text": [combined_reasoning]
            }
        },
        {
            "from_name": "human_input_vi_confidence",
            "to_name": "human_text",
            "type": "rating",
            "value": {
                "rating": human_input_confidence
            }
        },
        {
            "from_name": "response_output_vi_confidence",
            "to_name": "response_text",
            "type": "rating",
            "value": {
                "rating": response_output_confidence
            }
        }
    ]

    # Add the current prediction to the batch
    new_predictions_batch.append({
        "task_id": task_id,
        "prediction_result": prediction_result
    })
    processed_task_ids.add(task_id)

    # If the batch is full, save to file and submit
    if len(new_predictions_batch) >= BATCH_SIZE:
        print(f"Submitting batch of {len(new_predictions_batch)} predictions...")
        # Append new batch to existing data and save
        existing_predictions_data.extend(new_predictions_batch)
        with open(TEMP_PREDICTIONS_FILE, 'w', encoding='utf-8') as f:
            json.dump(existing_predictions_data, f, indent=2, ensure_ascii=False)

        # Submit to Label Studio one by one
        submitted_count = 0
        for item in new_predictions_batch:
            current_human_conf = next((res['value']['rating'] for res in item['prediction_result'] if res['from_name'] == 'human_input_vi_confidence'), 1)
            current_response_conf = next((res['value']['rating'] for res in item['prediction_result'] if res['from_name'] == 'response_output_vi_confidence'), 1)
            normalized_score = (current_human_conf + current_response_conf) / 10.0

            try:
                ls.predictions.create(
                    task=item['task_id'],
                    # project=259770, # IMPORTANT: Ensure this project ID is correct
                    model_version=api_config['model_name'], # Using the model name from config
                    score=normalized_score,
                    result=item['prediction_result'],
                )
                submitted_count += 1
            except Exception as e:
                print(f"Failed to create prediction for task {item['task_id']}: {e}")
        
        print(f"Successfully submitted {submitted_count} predictions from batch to Label Studio.")
        new_predictions_batch = [] # Clear the batch for the next set

# After the loop, submit any remaining predictions in the last batch
if new_predictions_batch:
    print(f"Submitting final batch of {len(new_predictions_batch)} predictions...")
    existing_predictions_data.extend(new_predictions_batch)
    with open(TEMP_PREDICTIONS_FILE, 'w', encoding='utf-8') as f:
        json.dump(existing_predictions_data, f, indent=2, ensure_ascii=False)

    submitted_count = 0
    for item in new_predictions_batch:
        current_human_conf = next((res['value']['rating'] for res in item['prediction_result'] if res['from_name'] == 'human_input_vi_confidence'), 1)
        current_response_conf = next((res['value']['rating'] for res in item['prediction_result'] if res['from_name'] == 'response_output_vi_confidence'), 1)
        normalized_score = (current_human_conf + current_response_conf) / 10.0

        try:
            ls.predictions.create(
                task=item['task_id'],
                project=259770,
                model_version=api_config['model_name'],
                score=normalized_score,
                result=item['prediction_result'],
            )
            submitted_count += 1
        except Exception as e:
            print(f"Failed to create prediction for task {item['task_id']}: {e}")

    print(f"Successfully submitted {submitted_count} predictions from final batch to Label Studio.")
    new_predictions_batch = []

print(f"\nAll predictions processed and submitted (or saved to {TEMP_PREDICTIONS_FILE}).")

No existing temporary predictions file found at temp_predictions.json. Starting fresh.
Starting translation and prediction generation for sample tasks...


Processing tasks:   0%|                                                                            | 0/5000 [00:00<?, ?it/s]

Processing tasks:   0%|                                                                 | 1/5000 [00:08<11:41:28,  8.42s/it]

Processing tasks:   0%|                                                                 | 2/5000 [00:15<10:59:24,  7.92s/it]

Processing tasks:   0%|                                                                 | 3/5000 [00:22<10:00:11,  7.21s/it]

...

Processing tasks:   5%|███                                                            | 243/5000 [35:01<13:03:10,  9.88s/it]

Processing tasks:   5%|███                                                            | 244/5000 [35:14<14:11:34, 10.74s/it]

Processing tasks:   5%|███                                                            | 245/5000 [35:24<13:53:21, 10.52s/it]

Processing tasks:   5%|███                                                            | 245/5000 [35:33<11:30:13,  8.71s/it]

KeyboardInterrupt: 